In [1]:
from TTS.api import TTS
import sounddevice as sd
import soundfile as sf

In [3]:
speak = "You’re talking about taking an existing wav file as input so that means you don’t want to generate synthetic audio, but instead load and run a WAV file through your program for playback, analysis, or maybe feeding it to a model"
speak1= "Good evening, Sir. All systems are online and fully operational."
speak3 = "Good morning. I am your personal AI assistant, designed to help you organize tasks, answer questions, and make complex information simple and clear. Today, I will demonstrate my ability to communicate in different voices, so you can imagine how I might sound if integrated into a real system."
tts = TTS(model_name="tts_models/en/vctk/vits", progress_bar=True)

 > tts_models/en/vctk/vits is already downloaded.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > initialization of speaker-embedding layers.


In [3]:
tts.tts_to_file(text=speak3, file_path="output.wav",speaker='p233' )
def play_wav(file_path: str):
    """Play a .wav file."""
    data, samplerate = sf.read(file_path)
    sd.play(data, samplerate)
    sd.wait()  # Wait until playback is finished


play_wav("output.wav")

 > Text splitted to sentences.
['Good morning.', 'I am your personal AI assistant, designed to help you organize tasks, answer questions, and make complex information simple and clear.', 'Today, I will demonstrate my ability to communicate in different voices, so you can imagine how I might sound if integrated into a real system.']
 > Processing time: 4.355021238327026
 > Real-time factor: 0.2671368515631563


In [11]:
def play_wav(file_path: str):
    """Play a .wav file."""
    data, samplerate = sf.read(file_path)
    sd.play(data, samplerate)
   # sd.wait()  # Wait until playback is finished
    print(data,samplerate)

play_wav("output.wav")

[-0.00192261 -0.00189209 -0.00241089 ...  0.          0.
  0.        ] 22050


In [ ]:
from TTS.api import TTS
import sounddevice as sd
import soundfile as sf

from ollama import chat
from ollama import ChatResponse
import time
tts = TTS(model_name="tts_models/en/vctk/vits", progress_bar=True)

def play_wav(file_path: str):
    """Play a .wav file."""
    data, samplerate = sf.read(file_path)
    sd.play(data, samplerate)
   # sd.wait()  # Wait until playback is finished
    print(data,samplerate)

in_chat = input("enter here")

response: ChatResponse = chat(model='llama3', messages=[
  {
    'role': 'user',
    'content': in_chat,
  },
])

tts.tts_to_file(text=response.message.content, file_path="ollama_tts.wav",speaker='p233' )

play_wav("ollama_tts.wav")

In [ ]:
import sounddevice as sd
import soundfile as sf
import queue
import json
from vosk import Model, KaldiRecognizer
from TTS.api import TTS
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# -------------------------------
# Load models
# -------------------------------
# Vosk STT
model_path = "/home/prot/Projects/Project-ALB/testing_modules/STT/model/vosk-model-en-in-0.5"
stt_model = Model(model_path)
rec = KaldiRecognizer(stt_model, 16000)

# Silero / VCTK TTS
tts = TTS(model_name="tts_models/en/vctk/vits", progress_bar=True)

# GPT-2 (small for local CPU)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2")
gpt_model.eval()

# -------------------------------
# Audio queue
# -------------------------------
q = queue.Queue()
def callback(indata, frames, time, status):
    if status:
        print(status)
    q.put(bytes(indata))

# -------------------------------
# Helper: play WAV
# -------------------------------
def play_wav(file_path: str):
    data, samplerate = sf.read(file_path)
    sd.play(data, samplerate)
    sd.wait()

# -------------------------------
# Helper: generate GPT response
# -------------------------------
def gpt_response(prompt: str, max_len=100):
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = gpt_model.generate(inputs, max_length=max_len, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# -------------------------------
# Run assistant
# -------------------------------
with sd.RawInputStream(samplerate=16000, blocksize=8000, dtype='int16',
                       channels=1, callback=callback):
    print("Listening... Speak now. Ctrl+C to stop.")
    try:
        while True:
            data = q.get()
            if rec.AcceptWaveform(data):
                res = json.loads(rec.Result())
                text = res.get("text", "")
                if text.strip():
                    print("You said:", text)
                    # Generate response
                    reply = gpt_response(text)
                    print("AI:", reply)
                    # TTS to WAV and play
                    tts.tts_to_file(text=reply, file_path="output.wav", speaker="p233")
                    play_wav("output.wav")
    except KeyboardInterrupt:
        print("Assistant stopped.")


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=13 max-active=7000 lattice-beam=6
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 1 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 2 orphan components.
LOG (VoskAPI:Collapse():nnet-utils.cc:1488) Added 1 components, removed 2
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from /home/prot/Projects/Project-ALB/testing_modules/STT/model/vosk-model-en-in-0.5/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:279) Loading HCLG from /home/prot/Projects/Project-ALB/testing_modules/STT/model/vosk-model-en-in-0.5/graph/HCLG.fst
LOG (VoskAPI:ReadDataFiles():model.cc:297) Loading words from /home/prot/Projects/Proj

 > tts_models/en/vctk/vits is already downloaded.
 > Using model: vits
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:1024
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:None
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1024
 > initialization of speaker-embedding layers.
